<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/image_visualization_with_slicer_desktop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run the full 3D Slicer desktop application in Colab with IDC data

---

## Summary

This notebook shows how to run the **complete [3D Slicer](https://www.slicer.org) desktop
application** — the real, full graphical program, not just its rendering engine — **inside a Google
Colab cell**, and use it to interactively explore a radiology image and its segmentation downloaded
from [NCI Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov).

3D Slicer is normally a desktop app. Here we launch the genuine application *headlessly* on the Colab
virtual machine (software OpenGL rendering into a virtual X display) and **stream its live GUI into
the notebook** over a WebSocket video connection using the [**desktopia**](https://github.com/pieper/desktopia)
project. The result is the actual Slicer interface — menus, module panel, slice views, 3D view,
mouse interaction — embedded in an output cell, with **no desktop install** on your machine.

As the example dataset we use the **same CT + segmentation** as the other IDC 3D-viewer tutorials
([ipyniivue](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/image_visualization_with_ipyniivue.ipynb)
and [trame-slicer](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/trame_slicer_visualization.ipynb)):
a low-dose chest CT from the [National Lung Screening Trial (NLST)](https://imaging.datacommons.cancer.gov/explore/?filters_for_load=collection_id_nlst)
together with its whole-body multi-organ [TotalSegmentator](https://github.com/wasserth/TotalSegmentator)
segmentation, published in IDC as the `TotalSegmentator-CT-Segmentations` analysis result.

**How this differs from the other two viewer tutorials**
* **ipyniivue** embeds a WebGL viewer (NiiVue) and needs the data converted to NIfTI first.
* **trame-slicer** embeds the 3D Slicer *rendering engine* through a custom trame web UI.
* **This notebook** streams the *entire 3D Slicer desktop application itself* — you get every Slicer
  module and the exact desktop UI, and Slicer loads the **DICOM image and DICOM SEG natively** (no
  conversion needed).

Upon completion of this tutorial you will learn how to:
* set up a headless Slicer environment in Colab and install the [QuantitativeReporting](https://github.com/QIICR/QuantitativeReporting) extension (which lets Slicer read DICOM SEG)
* download a DICOM image series and its DICOM Segmentation from IDC with [`idc-index`](https://github.com/ImagingDataCommons/idc-index)
* launch the real 3D Slicer application headlessly, load the DICOM data, and build 3D organ surfaces
* stream and interact with the live Slicer desktop directly inside a Colab cell

> **Runtime note.** A standard **CPU** Colab runtime is sufficient — rendering uses software OpenGL
> (Mesa/llvmpipe). The first two cells download ~400 MB (3D Slicer) and install system libraries, so
> the initial setup takes a few minutes.

> **Credit.** The headless-Slicer-in-Colab streaming setup used here is from Steve Pieper's
> [desktopia](https://github.com/pieper/desktopia) project; this notebook adapts its
> `desktopia_tcia_kidney` example to use IDC as the data source.

---
Initial version: Jun 2026

Updated: Jun 2026

## 1. Install system dependencies

3D Slicer is a Qt/VTK desktop application, so to run it headlessly we need a virtual X display
(`xvfb`), a software OpenGL stack (Mesa), a lightweight window manager, and the GStreamer + XCB
libraries that desktopia uses to capture and stream the GUI. This cell installs them.

In [ ]:
%%bash
set -e
cd /content
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq --no-install-recommends \
  xvfb xclip matchbox-window-manager fonts-dejavu-core libgl1-mesa-dri libglu1-mesa \
  gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly \
  python3-gi gir1.2-gstreamer-1.0 gir1.2-gst-plugins-base-1.0 python3-xlib \
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 libxcb-render-util0 libxcb-shape0 \
  libxcb-sync1 libxcb-xfixes0 libxcb-xinerama0 libxcb-xkb1 libxkbcommon-x11-0 libxcb-cursor0 libxcb-util1 \
  libodbc2 libpq5 libpulse-mainloop-glib0 libpcre2-16-0 \
  libxcomposite1 libxdamage1 libxtst6 libhwloc15 libnspr4 libnss3 >/dev/null
apt-get install -y -qq --no-install-recommends libasound2 libcups2 >/dev/null 2>&1 \
  || apt-get install -y -qq --no-install-recommends libasound2t64 libcups2t64 >/dev/null
pip install -q websockets aioquic
echo 'deps installed' 

## 2. Install 3D Slicer and the QuantitativeReporting extension

This cell clones the desktopia streaming server, downloads the latest 3D Slicer release (~400 MB)
into `/opt`, and installs the [**QuantitativeReporting**](https://github.com/QIICR/QuantitativeReporting)
extension. QuantitativeReporting provides the DICOM plugin that lets Slicer read **DICOM Segmentation
(SEG)** objects like the one we will load from IDC. The install runs Slicer once, headlessly, just to
add the extension.

In [ ]:
%%bash
set -e
cd /content
REPO=${DESKTOPIA_REPO:-https://github.com/pieper/desktopia}
BRANCH=${DESKTOPIA_BRANCH:-software-render}
rm -rf /content/desktopia
git clone -q --branch "$BRANCH" "$REPO" /content/desktopia || git clone -q "$REPO" /content/desktopia
if ! ls -d /opt/Slicer-*/ >/dev/null 2>&1; then
  echo 'downloading 3D Slicer (~400 MB)...'
  curl -L --retry 3 'https://download.slicer.org/download?os=linux&stability=release' | tar -xz -C /opt
fi
SDIR=$(ls -d /opt/Slicer-*/ | head -1); echo "Slicer: $SDIR"
cat > /tmp/install_qr.py <<'PY'
import slicer
emm = slicer.app.extensionsManagerModel()
emm.interactive = False
try: emm.updateExtensionsMetadataFromServer(True, True)
except Exception as e: print('metadata:', e, flush=True)
ok = False
try:
    emm.downloadAndInstallExtensionByName('QuantitativeReporting', True, True)
    ok = emm.isExtensionInstalled('QuantitativeReporting')
except Exception as e:
    print('install error:', e, flush=True)
print('QR_INSTALLED=', ok, flush=True)
slicer.util.exit(0 if ok else 1)
PY
echo 'installing QuantitativeReporting...'
LIBGL_ALWAYS_SOFTWARE=1 GALLIUM_DRIVER=llvmpipe HOME=/root \
  xvfb-run -a "$SDIR/Slicer" --no-splash --no-main-window --ignore-slicerrc \
  --python-script /tmp/install_qr.py || echo 'QR install returned nonzero (see log above)' 

## 3. Download a CT and its segmentation from IDC

We use [`idc-index`](https://github.com/ImagingDataCommons/idc-index) to download the data directly
from IDC — no TCIA / `tcia_utils` and no Google Cloud authentication required. For a fully
reproducible demo we pin the **same** image + segmentation pair used in the ipyniivue and trame-slicer
tutorials: a small NLST low-dose chest CT and its whole-body TotalSegmentator organ segmentation
(both licensed CC BY 4.0).

Both series are downloaded with a *flat* layout (`dirTemplate=""`) into one folder so that Slicer's
DICOM importer can ingest the image and the SEG together.

In [ ]:
%pip install -q --upgrade idc-index

import glob
from idc_index import IDCClient

client = IDCClient()
print("idc-index is using IDC data version:", client.get_idc_version())

# Same pinned pair as the other IDC 3D-viewer tutorials, for consistency:
ct_series  = "1.2.840.113654.2.55.71041873368734986406977154644223539362"  # NLST low-dose chest CT
seg_series = "1.2.276.0.7230010.3.1.3.313263360.84.1706324554.366745"      # TotalSegmentator SEG

client.download_from_selection(
    downloadDir="/content/idcDownload",
    seriesInstanceUID=[ct_series, seg_series],
    dirTemplate="",   # flat layout: all .dcm files directly under idcDownload/
)

print("DICOM files downloaded:", len(glob.glob("/content/idcDownload/*.dcm")))

## 4. Launch 3D Slicer and load the data

This cell starts the real Slicer application on a virtual X display and runs a small startup script
*inside* Slicer that:

1. imports everything under `/content/idcDownload` into a temporary DICOM database and loads it —
   Slicer reads the CT series **and** the DICOM SEG natively (the SEG becomes a segmentation node);
2. builds a **3D closed-surface representation** for every segment so the organs appear in the 3D view
   (the TotalSegmentator SEG has dozens of structures, so this takes a few seconds);
3. sets the four-up layout and frames the slice and 3D views.

desktopia then captures the GUI from the virtual display and serves it over a WebSocket video stream.

In [ ]:
import os, time, glob, pathlib, subprocess
os.chdir('/content'); WORK = '/content/desktopia'
WIDTH, HEIGHT, FPS, BITRATE = 1280, 720, 15, 4000

# STARTUP runs inside Slicer: load /content/idcDownload, overlay the SEG, build 3D surfaces, frame views.
STARTUP = r"""
import slicer
from DICOMLib import DICOMUtils
loaded = []
with DICOMUtils.TemporaryDICOMDatabase() as db:
    DICOMUtils.importDicom('/content/idcDownload', db)
    for p in db.patients():
        loaded += DICOMUtils.loadPatientByUID(p)
print('loaded nodes:', loaded, flush=True)
for seg in slicer.util.getNodesByClass('vtkMRMLSegmentationNode'):
    seg.CreateClosedSurfaceRepresentation()           # 3D surface of the segments
    dn = seg.GetDisplayNode()
    if dn:
        dn.SetVisibility(True); dn.SetVisibility3D(True)
slicer.app.layoutManager().setLayout(slicer.vtkMRMLLayoutNode.SlicerLayoutFourUpView)
vols = slicer.util.getNodesByClass('vtkMRMLScalarVolumeNode')
if vols:
    slicer.util.setSliceViewerLayers(background=vols[0], fit=True)
slicer.util.resetSliceViews()
try:
    tdv = slicer.app.layoutManager().threeDWidget(0).threeDView()
    tdv.resetFocalPoint(); tdv.resetCamera()
except Exception as e:
    print('3d reset:', e, flush=True)
"""

env = dict(os.environ, DISPLAY=':2', LIBGL_ALWAYS_SOFTWARE='1', GALLIUM_DRIVER='llvmpipe', HOME='/root')
for pat in ('server.py', 'SlicerApp-real'):
    subprocess.run(['pkill', '-f', pat], check=False)
for proc in ('Xvfb', 'matchbox-window-manager'):
    subprocess.run(['pkill', '-x', proc], check=False)
time.sleep(1)

subprocess.run('openssl req -x509 -newkey ec -pkeyopt ec_paramgen_curve:prime256v1 '
               '-keyout /tmp/k.pem -out /tmp/c.pem -days 1 -nodes -subj /CN=desktopia',
               shell=True, check=True, stderr=subprocess.DEVNULL)
subprocess.Popen(f'Xvfb :2 -screen 0 {WIDTH}x{HEIGHT}x24 +extension GLX +render -noreset',
                 shell=True, env=env, stdout=open('/tmp/xvfb.log','w'), stderr=subprocess.STDOUT)
for _ in range(80):
    if os.path.exists('/tmp/.X11-unix/X2'): break
    time.sleep(0.25)
subprocess.Popen('matchbox-window-manager -use_titlebar no', shell=True, env=env,
                 stdout=open('/tmp/wm.log','w'), stderr=subprocess.STDOUT)
time.sleep(1)

SDIR = sorted(glob.glob('/opt/Slicer-*/'))[0]
pathlib.Path('/tmp/slicer_startup.py').write_text(STARTUP)
subprocess.Popen(f'{SDIR}/Slicer --no-splash --python-script /tmp/slicer_startup.py',
                 shell=True, env=env, stdout=open('/tmp/slicer.log','w'), stderr=subprocess.STDOUT)

pathlib.Path(f'{WORK}/client/status.json').write_text('{"ready":true,"transport":"websocket"}')
subprocess.Popen('python3 server.py --cert /tmp/c.pem --key /tmp/k.pem '
                 f'--source xvfb --width {WIDTH} --height {HEIGHT} --fps {FPS} --bitrate {BITRATE} '
                 '--ws-plain --serve-dir client',
                 shell=True, env=env, cwd=WORK,
                 stdout=open('/tmp/server.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('--- server.log ---'); print(open('/tmp/server.log').read()[-1200:])
print('Slicer is loading the CT+SEG; the view appears below in a few seconds.')

## 5. View the live 3D Slicer desktop

Run the cell below to embed the streamed Slicer GUI in the notebook. Give it a few seconds to connect
and for the CT + segmentation to finish loading, then interact directly:

* **scroll** in a slice view to page through slices;
* **left-drag** in the 3D view to rotate, **scroll** to zoom;
* use the **module panel** on the left exactly as in desktop Slicer — every Slicer module is available
  (try *Segmentations*, *Volume Rendering*, or *Data*).

> If the view is blank, re-run this cell (the stream may still have been starting), or re-run the
> previous cell to relaunch Slicer.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(4434, path='/index.html', height=600, cache_in_notebook=False)

## 6. Cite the data you used

Most IDC data (including this collection) is licensed CC BY 4.0, which permits commercial use **with
attribution**. `idc-index` generates ready-to-use citations for any selection — here both the NLST
images and the TotalSegmentator analysis result.

In [ ]:
for citation in client.citations_from_selection(seriesInstanceUID=[ct_series, seg_series]):
    print(citation, "\n")

## Next steps

* **Use the full Slicer app.** Because this is the complete 3D Slicer desktop, you can do anything
  Slicer supports — edit the segmentation in *Segment Editor*, tune *Volume Rendering*, take
  measurements, or run other extensions.
* **Swap in different data.** The download in Section 3 works for any IDC series; change `ct_series`
  / `seg_series` to any image (CT, MR, PT, ...) and any DICOM SEG. Browse collections in the
  [IDC Portal](https://portal.imaging.datacommons.cancer.gov/explore/). Drop the segmentation to view
  an image alone.
* **Prefer a lighter-weight embedded viewer?** See the companion tutorials that render inside the cell
  without streaming a full desktop:
  [ipyniivue](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/image_visualization_with_ipyniivue.ipynb)
  (WebGL widget) and
  [trame-slicer](https://github.com/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/trame_slicer_visualization.ipynb)
  (Slicer rendering engine via trame).
* More tutorials are in the [IDC-Tutorials repository](https://github.com/ImagingDataCommons/IDC-Tutorials).

## Support

If you have any questions about this notebook, please post your question on the
[IDC User Forum](https://discourse.canceridc.dev) or
[open an issue](https://github.com/ImagingDataCommons/IDC-Tutorials/issues/new) in the
[IDC Tutorials repository](https://github.com/ImagingDataCommons/IDC-Tutorials).

For the Slicer-in-Colab streaming mechanism itself, see the
[desktopia repository](https://github.com/pieper/desktopia).

## Acknowledgments

Imaging Data Commons has been funded in whole or in part with Federal funds from the National Cancer
Institute, National Institutes of Health, under Task Order No. HHSN26110071 under Contract No.
HHSN261201500003I.

This notebook runs [3D Slicer](https://www.slicer.org) headlessly in Colab using Steve Pieper's
[desktopia](https://github.com/pieper/desktopia) project, with the
[QuantitativeReporting](https://github.com/QIICR/QuantitativeReporting) extension for DICOM SEG
support. The segmentation shown is from the IDC analysis result *TotalSegmentator-CT-Segmentations*
([doi:10.5281/zenodo.8347011](https://doi.org/10.5281/zenodo.8347011)), computed on the NLST collection.

If you use IDC in your research, please cite the following publication:

> Fedorov, A., Longabaugh, W. J. R., Pot, D., Clunie, D. A., Pieper, S. D., Gibbs, D. L., Bridge, C., Herrmann, M. D., Homeyer, A., Lewis, R., Aerts, H. J. W., Krishnaswamy, D., Thiriveedhi, V. K., Ciausu, C., Schacherer, D. P., Bontempi, D., Pihl, T., Wagner, U., Farahani, K., Kim, E. & Kikinis, R. _National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence_. RadioGraphics (2023). https://doi.org/10.1148/rg.230180